# Tornado Experiment — Master Runner

Executes the full analysis pipeline without visualizations:

1. **Load run data** — attribute tables + wide-format simulation output
2. **Post-processing** — intertemporal decomposition / rescaling
3. **Cost-benefits** — system + technical costs → `cost_benefits_data_tornado.csv`
4. **MAC analysis** — marginal abatement costs → `marginal_abatement_costs_tornado.csv`
5. **Tableau export** — tornado plot data → `Tableau/data/tableau_tornado.csv`

All configuration lives in `scripts/config.py`.  
Shared pipeline modules (`data_loading`, `postprocessing`, `cost_benefits_pipeline`) are imported from `../shared_scripts/`.


## 0 · Environment setup

In [1]:
import os
import sys
import pathlib
import logging
import warnings

%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
RUNNER_DIR  = pathlib.Path(os.getcwd()).resolve()
SCRIPTS_DIR = RUNNER_DIR / "scripts"
SHARED_DIR  = RUNNER_DIR.parent / "shared_scripts"

assert SCRIPTS_DIR.exists(), f"scripts/ not found at {SCRIPTS_DIR}"
assert SHARED_DIR.exists(),  f"shared_scripts/ not found at {SHARED_DIR}"

for p in (SCRIPTS_DIR, SHARED_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ── Config ────────────────────────────────────────────────────────────────────
import config as cfg

# Project root (needed for ssp_modeling.* imports inside pipeline scripts)
if str(cfg.PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.PROJECT_DIR))

# notebooks/ dir (needed for utils.logger_utils)
if str(cfg.NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.NOTEBOOKS_DIR))

from utils.logger_utils import setup_clean_logger, mute_external_loggers

logger = setup_clean_logger("tornado", logging.INFO)
mute_external_loggers(["sisepuede"])
logger.info("Environment ready.")
logger.info(f"Project dir : {cfg.PROJECT_DIR}")
logger.info(f"Run output  : {cfg.RUN_ID_OUTPUT_DIR}")

2026-04-14 09:29:24,175 - INFO - Environment ready.
2026-04-14 09:29:24,176 - INFO - Project dir : /Users/fabianfuentes/git/ssp_libya
2026-04-14 09:29:24,176 - INFO - Run output  : /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-13T21;07;08.365137


## 1 · Load run data

In [2]:
from data_loading import load_attribute_tables, parse_strategy_metadata, load_wide_export

# Attribute tables
att_primary, att_strategy = load_attribute_tables(cfg.RUN_ID_OUTPUT_DIR)
att_strategy = parse_strategy_metadata(att_strategy)

logger.info(f"att_primary  : {att_primary.shape}")
logger.info(f"att_strategy : {att_strategy.shape}")

# Wide-format simulation output
df_export = load_wide_export(cfg.RUN_ID_OUTPUT_DIR, cfg.PRIMARY_IDS_FILTER)
logger.info(f"df_export    : {df_export.shape}")

2026-04-14 09:29:24,203 - INFO - att_primary  : (30, 4)
2026-04-14 09:29:24,204 - INFO - att_strategy : (133, 10)
2026-04-14 09:29:24,452 - INFO - df_export    : (1080, 4107)


## 2 · Post-processing — intertemporal decomposition

In [3]:
from postprocessing import run_decomposition

df_decomposed = run_decomposition(
    df_export    = df_export,
    project_dir  = cfg.PROJECT_DIR,
    targets_path = cfg.TARGETS_PATH,
    iso_code3    = cfg.ISO_CODE3,
    year_ref     = cfg.YEAR_REF,
    region       = cfg.REGION,
    output_path  = cfg.OUTPUT_DECOMPOSED,
)
logger.info(f"df_decomposed : {df_decomposed.shape}  → {cfg.OUTPUT_DECOMPOSED}")

Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_biomass (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_coal (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_coal_ccs (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_gas_ccs (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_geothermal (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_hydropower (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_nuclear (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_ocean (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_solar (time_period == 8)
Changed 30 zero(s) in: emission_co2e_ch4_entc_generation_pp_wind (time_period == 8)
Changed 30 zero(s) in: emission_co2e_co2_entc_generation_pp_coal (time_period == 8)
Changed 30 zero(s) in: emission_co2e_co2_entc_gen

2026-04-14 09:29:28,861 - INFO - df_decomposed : (840, 4107)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-13T21;07;08.365137/decomposed_ssp_output_tornado.csv


Decomposed output written to: /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-13T21;07;08.365137/decomposed_ssp_output_tornado.csv
Done: libya


## 3 · Cost-benefits

In [4]:
from cost_benefits_pipeline import run_cost_benefits

cb_data = run_cost_benefits(
    df_decomposed      = df_decomposed,
    att_primary        = att_primary,
    att_strategy       = att_strategy,
    cb_config_path     = cfg.CB_CONFIG_PATH,
    run_output_dir     = cfg.RUN_ID_OUTPUT_DIR,
    project_dir        = cfg.PROJECT_DIR,
    strategy_code_base = cfg.STRATEGY_CODE_BASE,
    output_path        = cfg.OUTPUT_CB_DATA,
)
logger.info(f"cb_data : {cb_data.shape}  → {cfg.OUTPUT_CB_DATA}")

Loading configuration from Excel file (fast path)
Database updated

************************************
*Strategy : PFLO:CONDITIONAL (0/29)
************************************


************************************
*Strategy : TORNADO_BASE:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_CONDITIONAL (1/29)
************************************


************************************
*Strategy : TORNADO_BASE:TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_CONDITIONAL (2/29)
************************************


************************************
*Strategy : TORNADO_BASE:TX:ENTC:DEC_LOSSES_STRATEGY_CONDITIONAL (3/29)
************************************


************************************
*Strategy : TORNADO_BASE:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_CONDITIONAL (4/29)
************************************


************************************
*Strategy : TORNADO_BASE:TX:FGTV:DEC_LEAKS_STRATEGY_CONDITIONAL (5/29)
************************************


************************************
*S

2026-04-14 09:29:47,865 - INFO - cb_data : (138768, 21)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-13T21;07;08.365137/cost_benefits_data_tornado.csv


## 4 · Marginal Abatement Cost (MAC)

In [5]:
from mac_pipeline import run_mac_analysis

mac_df = run_mac_analysis(
    df_decomposed           = df_decomposed,
    cb_data                 = cb_data,
    att_primary             = att_primary,
    att_strategy            = att_strategy,
    iso_code3               = cfg.ISO_CODE3,
    region                  = cfg.REGION,
    invent_dir              = cfg.INVENT_DIR,
    targets_path            = cfg.TARGETS_PATH,
    run_output_dir          = cfg.RUN_ID_OUTPUT_DIR,
    strategy_code_baseline  = cfg.STRATEGY_CODE_BASELINE,
    output_filename         = cfg.OUTPUT_MAC.name,
)
logger.info(f"mac_df : {mac_df.shape}  → {cfg.OUTPUT_MAC}")

2026-04-14 09:29:47,990 - INFO - mac_df : (29, 9)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-13T21;07;08.365137/marginal_abatement_costs_tornado.csv


## 5 · Tableau export

In [6]:
from mac_pipeline import build_attribute_map

att_map = build_attribute_map(
    att_primary    = att_primary,
    att_strategy   = att_strategy,
    run_output_dir = cfg.RUN_ID_OUTPUT_DIR,
)
logger.info(f"att_map : {att_map.shape}  → {cfg.RUN_ID_OUTPUT_DIR / 'ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv'}")

2026-04-14 09:29:48,003 - INFO - att_map : (29, 8)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-13T21;07;08.365137/ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv


In [7]:
from mac_pipeline import build_tableau_tornado

tableau_tornado = build_tableau_tornado(
    mac_df         = mac_df,
    run_output_dir = cfg.RUN_ID_OUTPUT_DIR,
    tableau_dir    = cfg.TABLEAU_DIR,
    output_path    = cfg.OUTPUT_TABLEAU_TORNADO,
)
logger.info(f"tableau_tornado : {tableau_tornado.shape}  → {cfg.OUTPUT_TABLEAU_TORNADO}")

2026-04-14 09:29:48,015 - INFO - tableau_tornado : (29, 14)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/Tableau/data/tableau_tornado.csv


## 6 · Summary

In [8]:
import pandas as pd

summary = pd.DataFrame([
    {"output": "decomposed SSP",    "rows": len(df_decomposed), "cols": df_decomposed.shape[1], "file": cfg.OUTPUT_DECOMPOSED.name},
    {"output": "cost-benefits data","rows": len(cb_data),       "cols": cb_data.shape[1],       "file": cfg.OUTPUT_CB_DATA.name},
    {"output": "MAC curves",        "rows": len(mac_df),        "cols": mac_df.shape[1],        "file": cfg.OUTPUT_MAC.name},
])
print(summary.to_string(index=False))

            output   rows  cols                                 file
    decomposed SSP    840  4107    decomposed_ssp_output_tornado.csv
cost-benefits data 138768    21       cost_benefits_data_tornado.csv
        MAC curves     29     9 marginal_abatement_costs_tornado.csv
